In [4]:
import torch
import torch.nn as nn
import numpy as np

CHARACTERS = [
    'Intervention', 'Barrier', 'CrossingSignal',
    'Man', 'Woman', 'Pregnant', 'Stroller', 'OldMan', 'OldWoman',
    'Boy', 'Girl', 'Homeless', 'LargeWoman', 'LargeMan', 'Criminal',
    'MaleExecutive', 'FemaleExecutive', 'FemaleAthlete', 'MaleAthlete',
    'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat'
]

CHAR_TO_IDX = {char: idx for idx, char in enumerate(CHARACTERS)}

class MoralReasoningTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_characters = 23
        self.embed_dim = 128

        self.character_embedding = nn.Embedding(23, 128)
        self.cardinality_embedding = nn.Embedding(11, 128)
        self.team_embedding = nn.Embedding(2, 128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128, nhead=8, dim_feedforward=512,
            dropout=0.05, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)
        self.cls_token = nn.Parameter(torch.randn(1, 1, 128))

        self.classifier = nn.Sequential(
            nn.LayerNorm(128),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(64, 1)
        )

    def encode_outcome(self, counts, team_id):
        batch_size = counts.shape[0]
        character_ids = torch.arange(23, device=counts.device).unsqueeze(0).expand(batch_size, -1)

        char_emb = self.character_embedding(character_ids)
        card_emb = self.cardinality_embedding(counts)
        team_emb = self.team_embedding(torch.full((batch_size, 23), team_id, device=counts.device, dtype=torch.long))

        return char_emb + card_emb + team_emb

    def forward(self, scenarios):
        batch_size = scenarios.shape[0]
        tokens_0 = self.encode_outcome(scenarios[:, 0, :], team_id=0)
        tokens_1 = self.encode_outcome(scenarios[:, 1, :], team_id=1)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        all_tokens = torch.cat([cls_tokens, tokens_0, tokens_1], dim=1)

        encoded = self.transformer(all_tokens)
        return self.classifier(encoded[:, 0, :])


def load_model(path='best_model.pt'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MoralReasoningTransformer()
    model.load_state_dict(torch.load(path, map_location=device)['model_state_dict'])
    model.to(device).eval()
    return model, device


def get_probs(model, device, scenario_tuple):
    """
    scenario_tuple: (dict_0, dict_1) where dicts have character names as keys
    Returns: [prob_outcome_0, prob_outcome_1] that sum to 1.0
    """
    outcome_0, outcome_1 = scenario_tuple

    vec_0 = np.zeros(23, dtype=np.int64)
    vec_1 = np.zeros(23, dtype=np.int64)

    for char, count in outcome_0.items():
        vec_0[CHAR_TO_IDX[char]] = count
    for char, count in outcome_1.items():
        vec_1[CHAR_TO_IDX[char]] = count

    scenario = torch.tensor([[vec_0, vec_1]], dtype=torch.long).to(device)

    with torch.no_grad():
        prob_1 = torch.sigmoid(model(scenario)).item()
        return [1.0 - prob_1, prob_1]


# Usage:
# model, device = load_model('best_model.pt')
# prob = get_probs(model, device, ({'Man': 3}, {'Criminal': 3}))

In [9]:
if __name__ == "__main__":
    # Load model
    model, device = load_model('best_model.pt')

    # Example 1: Save 3 men and 1 dog vs 2 women and 1 criminal
    scenario1 = (
        {'Woman': 2, 'Criminal': 1},
        {'Man': 3, 'Dog': 1}
    )

    prob1 = get_probs(model, device, scenario1)
    print(f"\nScenario 1:")
    print(f"  Outcome 0: {scenario1[0]}")
    print(f"  Outcome 1: {scenario1[1]}")
    print(f"  P(choose outcome 1): {prob1}")

    # Example 2: Save 5 men vs 5 criminals
    scenario2 = (
        {'Man': 5},
        {'Criminal': 5}
    )

    prob2 = get_probs(model, device, scenario2)
    print(f"\nScenario 2:")
    print(f"  Outcome 0: {scenario2[0]}")
    print(f"  Outcome 1: {scenario2[1]}")
    print(f"  P(choose outcome 1): {prob2}")

    # Example 3: Save 1 pregnant woman vs 2 old men
    scenario3 = (
        {'Pregnant': 1},
        {'OldMan': 2}
    )

    prob3 = get_probs(model, device, scenario3)
    print(f"\nScenario 3:")
    print(f"  Outcome 0: {scenario3[0]}")
    print(f"  Outcome 1: {scenario3[1]}")
    print(f"  P(choose outcome 1): {prob3}")

    # Example 4: Intervention scenario - crossing signal
    scenario4 = (
        {'CrossingSignal': 1, 'Man': 3},
        {'Barrier': 1, 'Woman': 2, 'Boy': 1}
    )

    prob4 = get_probs(model, device, scenario4)
    print(f"\nScenario 4:")
    print(f"  Outcome 0: {scenario4[0]}")
    print(f"  Outcome 1: {scenario4[1]}")
    print(f"  P(choose outcome 1): {prob4}")


Scenario 1:
  Outcome 0: {'Woman': 2, 'Criminal': 1}
  Outcome 1: {'Man': 3, 'Dog': 1}
  P(choose outcome 1): [0.12127017974853516, 0.8787298202514648]

Scenario 2:
  Outcome 0: {'Man': 5}
  Outcome 1: {'Criminal': 5}
  P(choose outcome 1): [0.8300654143095016, 0.16993458569049835]

Scenario 3:
  Outcome 0: {'Pregnant': 1}
  Outcome 1: {'OldMan': 2}
  P(choose outcome 1): [0.9416625164449215, 0.058337483555078506]

Scenario 4:
  Outcome 0: {'CrossingSignal': 1, 'Man': 3}
  Outcome 1: {'Barrier': 1, 'Woman': 2, 'Boy': 1}
  P(choose outcome 1): [0.5341806411743164, 0.4658193588256836]


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
